# Module 4 — RetailPulse Recommendation Engine

## Objective
Build a basket-based product recommender using the association rules exported by Module 3. Given one or more products already in a basket, the engine finds rules whose antecedents are fully present and ranks new product suggestions using **confidence, lift, and support**.

This is an association-rule recommender, not a customer-personalisation model. It does not use customer identity, segmentation labels, or inactivity scores in this module.

### Inputs
- `artifacts/association/association_rules.csv` — generated by Module 3.
- `data/processed/cleaned_transactions.csv` — used only to map stock codes to readable product descriptions.

### Outputs
- `artifacts/recommendation/recommendation_engine_sample.csv` — example recommendations from a reproducible sample basket.
- `artifacts/recommendation/recommendation_engine_metadata.json` — input/output details and recommendation logic.

## 1. Imports and project paths
Run this notebook from the RetailPulse project. The path helper checks the current directory and its parents so it also works when Jupyter starts inside `notebooks/`.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed" / "cleaned_transactions.csv").exists() and (candidate / "artifacts" / "association").exists():
            return candidate
    raise FileNotFoundError("Could not locate RetailPulse project root. Run this notebook inside the RetailPulse project.")

PROJECT_ROOT = find_project_root(Path.cwd())
RULES_PATH = PROJECT_ROOT / "artifacts" / "association" / "association_rules.csv"
TRANSACTIONS_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "recommendation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Association rules exist:", RULES_PATH.exists())
print("Transactions file exists:", TRANSACTIONS_PATH.exists())
print("Output directory:", OUTPUT_DIR)

Project root: c:\Users\Mehak\OneDrive\MEHAK\Desktop\RetailPulse
Association rules exist: True
Transactions file exists: True
Output directory: c:\Users\Mehak\OneDrive\MEHAK\Desktop\RetailPulse\artifacts\recommendation


## 2. Load Module 3 rules and product descriptions
Stock codes are read as strings to preserve identifiers. Product descriptions are used for display only; matching is performed on stock codes.

In [2]:
if not RULES_PATH.exists():
    raise FileNotFoundError(f"Missing {RULES_PATH}. Run Module 3 and export association_rules.csv first.")
if not TRANSACTIONS_PATH.exists():
    raise FileNotFoundError(f"Missing {TRANSACTIONS_PATH}.")

rules = pd.read_csv(RULES_PATH, dtype={"antecedent_codes": "string", "consequent_codes": "string"})
product_data = pd.read_csv(TRANSACTIONS_PATH, usecols=["StockCode", "Description"], dtype={"StockCode": "string"})

required_rule_columns = {"antecedent_codes", "consequent_codes", "support", "confidence", "lift"}
missing_columns = required_rule_columns - set(rules.columns)
if missing_columns:
    raise ValueError(f"Association rules file is missing columns: {sorted(missing_columns)}")

product_data = product_data.dropna(subset=["StockCode", "Description"]).copy()
product_data["StockCode"] = product_data["StockCode"].str.strip()
product_data["Description"] = product_data["Description"].str.strip()
product_data = product_data[product_data["StockCode"].ne("")]
product_data = product_data.drop_duplicates(subset="StockCode", keep="first")
code_to_name = dict(zip(product_data["StockCode"], product_data["Description"]))
name_to_codes = {}
for stock_code, description in code_to_name.items():
    name_to_codes.setdefault(description.casefold(), []).append(stock_code)

print(f"Association rules loaded: {len(rules):,}")
print(f"Product descriptions mapped: {len(code_to_name):,}")
display(rules.head())

Association rules loaded: 1,344
Product descriptions mapped: 4,916


,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_codes,consequent_codes
0,0.013898,0.014073,0.010205,0.734291,52.177614,1.0,0.010010,3.710550,0.994659,0.574438,0.730498,0.729734,22745 | 22748,22746
1,0.014073,0.013898,0.010205,0.725177,52.177614,1.0,0.010010,3.588138,0.994835,0.574438,0.721304,0.729734,22746,22745 | 22748
2,0.011827,0.017541,0.010205,0.862869,49.190909,1.0,0.009998,7.164392,0.991397,0.532552,0.860421,0.722331,22746 | 22748,22745
3,0.017541,0.011827,0.010205,0.581792,49.190909,1.0,0.009998,2.362876,0.997163,0.532552,0.576787,0.722331,22745,22746 | 22748
4,0.011503,0.018514,0.010205,0.887202,47.919655,1.0,0.009992,8.701248,0.990526,0.515113,0.885074,0.719207,22745 | 22746,22748


## 3. Prepare rule antecedents and consequents
The exported CSV stores itemsets as pipe-separated stock codes. Convert these strings into sets so basket matching is reliable.

In [3]:
def parse_item_codes(value):
    if pd.isna(value):
        return frozenset()
    return frozenset(part.strip() for part in str(value).split("|") if part.strip())

rules = rules.copy()
rules["antecedent_set"] = rules["antecedent_codes"].apply(parse_item_codes)
rules["consequent_set"] = rules["consequent_codes"].apply(parse_item_codes)
for metric in ["support", "confidence", "lift"]:
    rules[metric] = pd.to_numeric(rules[metric], errors="coerce")
rules = rules.dropna(subset=["support", "confidence", "lift"])
rules = rules[(rules["antecedent_set"].map(len) > 0) & (rules["consequent_set"].map(len) > 0)]
print(f"Usable rules: {len(rules):,}")
print("Antecedent sizes:")
display(rules["antecedent_set"].map(len).value_counts().sort_index().rename_axis("number_of_products").to_frame("rules"))

Usable rules: 1,344
Antecedent sizes:


,rules
number_of_products,
1,1032
2,312


## 4. Product input resolver
The recommender accepts either stock codes or exact product descriptions. If a description maps to multiple stock codes, it asks for the stock code to avoid silently choosing the wrong product.

In [4]:
def resolve_product(value):
    value = str(value).strip()
    if value in code_to_name:
        return value
    matches = name_to_codes.get(value.casefold(), [])
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError(f"Description is ambiguous; use a StockCode instead: {value} -> {matches[:10]}")
    raise ValueError(f"Product not found by StockCode or exact Description: {value}")


def product_label(stock_code):
    return code_to_name.get(stock_code, "Unknown product")

## 5. Recommendation function

For an input basket, a rule is eligible when **all products in its antecedent are already in the basket**. Products already in the basket are removed from the suggestions. If multiple rules suggest the same product, the best available confidence/lift/support values are retained for ranking.

Ranking is lexicographic: confidence first, then lift, then support. These metrics are displayed separately so the recommendation remains interpretable.

In [5]:
def recommend_products(basket, top_n=5):
    """Recommend products from association rules for a basket of stock codes or exact descriptions."""
    if top_n < 1:
        raise ValueError("top_n must be at least 1")
    if isinstance(basket, (str, int, float)):
        basket = [basket]
    basket_codes = {resolve_product(item) for item in basket}
    if not basket_codes:
        raise ValueError("Provide at least one product in the basket.")

    matched = rules[rules["antecedent_set"].apply(lambda items: items.issubset(basket_codes))].copy()
    if matched.empty:
        return pd.DataFrame(columns=["recommended_stock_code", "recommended_product", "confidence", "lift", "support", "matched_antecedent"])

    rows = []
    for _, rule in matched.iterrows():
        for candidate in rule["consequent_set"] - basket_codes:
            rows.append({
                "recommended_stock_code": candidate,
                "confidence": float(rule["confidence"]),
                "lift": float(rule["lift"]),
                "support": float(rule["support"]),
                "matched_antecedent": " | ".join(sorted(rule["antecedent_set"]))
            })

    if not rows:
        return pd.DataFrame(columns=["recommended_stock_code", "recommended_product", "confidence", "lift", "support", "matched_antecedent"])

    candidates = pd.DataFrame(rows)
    candidates = candidates.sort_values(
        ["confidence", "lift", "support"], ascending=False
    ).drop_duplicates(subset="recommended_stock_code", keep="first")
    candidates["recommended_product"] = candidates["recommended_stock_code"].map(product_label)
    columns = ["recommended_stock_code", "recommended_product", "confidence", "lift", "support", "matched_antecedent"]
    return candidates[columns].head(top_n).reset_index(drop=True)

## 6. Run a reproducible example
Use the antecedent of the highest-lift rule as a demonstration basket. This guarantees that the example basket is represented in the mined rule set. It is a technical demonstration, not an evaluation on unseen customer data.

In [6]:
best_rule = rules.sort_values(["lift", "confidence", "support"], ascending=False).iloc[0]
sample_basket_codes = sorted(best_rule["antecedent_set"])
print("Example basket:")
for code in sample_basket_codes:
    print(f"- {code}: {product_label(code)}")

sample_recommendations = recommend_products(sample_basket_codes, top_n=10)
print("\nRecommended products:")
display(sample_recommendations)

assert not set(sample_recommendations["recommended_stock_code"]).intersection(sample_basket_codes)
print("Validation passed: no recommended item is already in the input basket.")

Example basket:
- 22745: POPPY'S PLAYHOUSE BEDROOM
- 22748: POPPY'S PLAYHOUSE KITCHEN

Recommended products:


,recommended_stock_code,recommended_product,confidence,lift,support,matched_antecedent
0,22746,POPPY'S PLAYHOUSE LIVINGROOM,0.734291,52.177614,0.010205,22745 | 22748


Validation passed: no recommended item is already in the input basket.


## 7. Try your own basket
Replace the example values with one or more stock codes or exact product descriptions from the dataset.

In [7]:
# Example: use StockCodes from your own invoice or product descriptions.
my_basket = sample_basket_codes
my_recommendations = recommend_products(my_basket, top_n=5)
display(my_recommendations)

,recommended_stock_code,recommended_product,confidence,lift,support,matched_antecedent
0,22746,POPPY'S PLAYHOUSE LIVINGROOM,0.734291,52.177614,0.010205,22745 | 22748


## 8. Export recommendations and metadata
The sample output is saved for inspection and for integration testing. The recommendation function itself can be imported/reused by a later application layer.

In [8]:
sample_output_path = OUTPUT_DIR / "recommendation_engine_sample.csv"
metadata_path = OUTPUT_DIR / "recommendation_engine_metadata.json"

sample_recommendations.to_csv(sample_output_path, index=False)
metadata = {
    "module": "RetailPulse Module 4 - Recommendation Engine",
    "method": "association_rule_based_basket_recommendation",
    "rules_input": str(RULES_PATH.relative_to(PROJECT_ROOT)),
    "product_mapping_input": str(TRANSACTIONS_PATH.relative_to(PROJECT_ROOT)),
    "rules_used": int(len(rules)),
    "ranking_order": ["confidence_descending", "lift_descending", "support_descending"],
    "matching_logic": "all antecedent products must be present in the input basket",
    "recommendations_exclude_products_already_in_basket": True,
    "sample_basket_stock_codes": sample_basket_codes,
    "sample_recommendations_export": str(sample_output_path.relative_to(PROJECT_ROOT))
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved:")
print("-", sample_output_path.relative_to(PROJECT_ROOT))
print("-", metadata_path.relative_to(PROJECT_ROOT))

Saved:
- artifacts\recommendation\recommendation_engine_sample.csv
- artifacts\recommendation\recommendation_engine_metadata.json


## 9. Interpretation and limitations

- A recommendation is produced because an association rule's antecedent is contained in the current basket; this indicates co-occurrence, not causation.
- Confidence estimates how often the consequent appeared when the antecedent appeared in the mined transactions.
- Lift compares that confidence with the consequent's overall frequency. Lift greater than 1 indicates a positive association.
- Support indicates the proportion of invoice baskets containing the complete rule itemset.
- The current recommender is basket-based. It does not yet personalise suggestions by customer profile, segment, inactivity probability, or time.
- If no rule matches the supplied basket, the function returns an empty result rather than inventing recommendations.
- The example generated here demonstrates function behaviour; it is not a formal offline accuracy evaluation. A separate train/test evaluation would be needed before claiming recommendation quality.

**Module 4 output:** a reusable `recommend_products()` function plus sample recommendation and metadata exports.